In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from math import radians, sin, cos, sqrt, atan2

# Load the dataset
data = pd.read_csv('uber.csv')

# Drop any irrelevant columns and handle missing data
data = data.drop(columns=['Unnamed: 0'], errors='ignore')  # Drop index column if exists
data = data.dropna()  # Drop rows with any missing values

# Remove rows with zero latitude and longitude
data = data[(data['pickup_longitude'] != 0) & (data['pickup_latitude'] != 0) &
            (data['dropoff_longitude'] != 0) & (data['dropoff_latitude'] != 0)]

# Define Haversine function for calculating distance
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Calculate distance
data['distance_km'] = data.apply(lambda row: haversine(row['pickup_latitude'], row['pickup_longitude'],
                                                      row['dropoff_latitude'], row['dropoff_longitude']), axis=1)

# Extract features from pickup_datetime
data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'], errors='coerce')
data = data.dropna(subset=['pickup_datetime'])  # Drop rows where datetime conversion failed
data['hour'] = data['pickup_datetime'].dt.hour
data['day_of_week'] = data['pickup_datetime'].dt.dayofweek

# Select features and target
X = data[['distance_km', 'hour', 'day_of_week', 'passenger_count']]
y = data['fare_amount']


In [3]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [4]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [5]:
# Initialize models
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)

# Train models
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

# Make predictions
rf_preds = rf_model.predict(X_test)
gb_preds = gb_model.predict(X_test)

# Evaluate performance
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = mean_squared_error(y_test, rf_preds, squared=False)
gb_mae = mean_absolute_error(y_test, gb_preds)
gb_rmse = mean_squared_error(y_test, gb_preds, squared=False)

print(f"Random Forest MAE: {rf_mae}")
print(f"Random Forest RMSE: {rf_rmse}")
print(f"Gradient Boosting MAE: {gb_mae}")
print(f"Gradient Boosting RMSE: {gb_rmse}")


Random Forest MAE: 2.554845129536714
Random Forest RMSE: 5.19813593418748
Gradient Boosting MAE: 2.334412152848877
Gradient Boosting RMSE: 4.914045327117488


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
